In [1]:
from src.retrieval.retriever import (
    _vector_search, _fulltext_search,
    _reciprocal_rank_fusion, _rerank
)
from sentence_transformers import SentenceTransformer
import psycopg2, os
from dotenv import load_dotenv

Loading retrieval models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Retrieval models ready.


In [2]:
load_dotenv()

EMBED_MODEL = SentenceTransformer("BAAI/bge-base-en-v1.5")
CONN        = psycopg2.connect(os.getenv("DATABASE_URL"))

query  = "What are Microsoft's AI and machine learning risks?"
ticker = "MSFT"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
# Step 1 — see raw vector results
qvec   = EMBED_MODEL.encode(query, normalize_embeddings=True).tolist()
v_res  = _vector_search(qvec, ticker, 10, CONN)
print("=== Vector search top 5 ===")
for r in v_res[:5]:
    print(f"  score={r['vector_score']:.4f} | {r['section']} | {r['text'][:80]}...")


=== Vector search top 5 ===
  score=0.7531 | preamble | Issues in the development, deployment, and use of AI may result in reputational ...
  score=0.6836 | preamble | AI is fundamentally transforming productivity for every individual, organization...
  score=0.6704 | preamble | We may not be able to protect information in our products and services from use ...
  score=0.6657 | preamble | Our internal environment continues to evolve. Often, we are early adopters of ne...
  score=0.6629 | preamble | Some companies compete with us by modifying and then distributing open-source so...


In [9]:
# Step 2 — see raw FTS results
f_res  = _fulltext_search(query, ticker, 10, CONN)
print("\n=== Full-text search top 5 ===")
print(f_res)
for r in f_res[:5]:
    print(f"  score={r['fts_score']:.5f} | {r['section']} | {r['text'][:80]}...")


=== Full-text search top 5 ===
[]


In [6]:
# Step 3 — see what RRF changes
merged = _reciprocal_rank_fusion(v_res, f_res)
print("\n=== After RRF merge top 5 ===")
for r in merged[:5]:
    print(f"  rrf={r['rrf_score']:.6f} | {r['section']} | {r['text'][:80]}...")


=== After RRF merge top 5 ===
  rrf=0.016393 | preamble | Issues in the development, deployment, and use of AI may result in reputational ...
  rrf=0.016129 | preamble | AI is fundamentally transforming productivity for every individual, organization...
  rrf=0.015873 | preamble | We may not be able to protect information in our products and services from use ...
  rrf=0.015625 | preamble | Our internal environment continues to evolve. Often, we are early adopters of ne...
  rrf=0.015385 | preamble | Some companies compete with us by modifying and then distributing open-source so...


In [9]:
# Step 4 — see what reranking changes
final  = _rerank(query, merged[:10], 5)
print("\n=== After reranking top 5 ===")
for r in final:
    print(f"  rerank={r['rerank_score']:.4f} | {r['section']} | {r['text'][:80]}...")


=== After reranking top 5 ===
  rerank=-1.2989 | item_1a | Legal and Regulatory Compliance Risks
The Company’s business, results of operati...
  rerank=-3.5766 | item_1a | Apple Inc. | 2025 Form 10-K | 12
The outcome of litigation or government investi...
  rerank=-3.9589 | item_1a | Apple Inc. | 2025 Form 10-K | 11
As with all companies, the security the Company...
  rerank=-5.9054 | item_1a | Adverse economic conditions can also lead to increased credit and collectibility...
  rerank=-7.2331 | item_1a | Apple Inc. | 2025 Form 10-K | 6
Following any interruption to its business, the ...
